# Generate AIND schema compliant AIND metadata for version 2.0

### Goal is to build a flexible extractor that accepts values that are dynamics -- requires a config file with metadata (just edit this). Map relevant data from rig and session generated from bonsai 

### NOTE: Project ID is Delphi

### Brandon Pratt, 12/03/2025

#### Upgrade aind-meta-data packages

In [43]:
pip install --upgrade aind-data-schema


  Attempting uninstall: aind-data-schema
    Found existing installation: aind-data-schema 2.2.0
    Uninstalling aind-data-schema-2.2.0:
      Successfully uninstalled aind-data-schema-2.2.0
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
pip install --upgrade aind-data-schema-models

In [44]:
# Import libraries
from datetime import date, datetime, timezone
from aind_data_schema_models.harp_types import HarpDeviceType
from aind_data_schema_models.modalities import Modality
from aind_data_schema_models.organizations import Organization
from aind_data_schema.components.coordinates import CoordinateSystemLibrary, Scale
from aind_data_schema_models.units import (
    FrequencyUnit,
    SizeUnit,
)
import pathlib
import json
import os
import math
import time
import msvcrt
import ctypes
import re
import numpy as np
from typing import Dict, Any, List, Tuple
from docx import Document
import pandas as pd
import unicodedata


### Create Pirouette + Delphi instrument.json extractor (seperate below)

In [45]:
# import the relevant device classes used in the setup
from aind_data_schema.components.devices import (
    Camera,
    CameraAssembly,
    DAQChannel,
    Device,
    Enclosure,
    EphysAssembly,
    EphysProbe,
    HarpDevice,
    Lens,
    LightEmittingDiode,
    Manipulator,
    Olfactometer,
    OlfactometerChannel,
    OlfactometerChannelType,
    OpenEphysAcquisitionBoard,
    ProbePort,
    Software,
    Computer,
)
from aind_data_schema.core.instrument import Instrument
from aind_data_schema_models.coordinates import AnatomicalRelative
from aind_data_schema_models.devices import CameraTarget
from aind_data_schema.components.connections import Connection


#### Config revisions

1. Add dilutions to rule
2. Add explicit odor names
3. Refactor Delphi configs to use aind-behavior-services (ruleSettines -> TaskLogic, hardwareSettings -> Rig.json, and create Session.json)

In [ ]:
# Imput parameters (will be required inputs for launcher)
experiment_types = [
    "delphi",
    "delphi_pirouette",
    "pirouette",
]  # Type of experiment being processed

# Description of experiments
expt_descriptions = {
    "delphi": "Behavior only recording with odor manipulation",
    "delphi_pirouette": "Chronic recording with odor manipulation",
    "pirouette": "Longterm chronic recording",
}


expt_idx = 2
current_experiment = experiment_types[expt_idx]  # Select experiment type to process
acquisition_type = expt_descriptions[current_experiment]
probe_id = "Probe B"
experiment_room = "157"
today = date.today()
instrument = "Chronic1"
delphi_computer_id = "DTMZ0334PS"
subject = "815154"  # "835602"  # "801055"
protocol = "2413"
experimenters = ["Brandon Pratt"]
surgeons = ["Carl Schoonover", "Ben Ouellette"]

# Path to surgery notes file
surgery_notes_path = pathlib.Path(r"\\allen\aind\scratch\chronos\surgeryNotes")
subject_notes = surgery_notes_path.joinpath(subject, f"{subject}_craniotomy-implantation.docx")

# path to config files
dataset_root = pathlib.Path(r"\\allen\aind\stage\chronic\data\2025-07-17T00-34-17")
metadata_path = dataset_root.joinpath("behavior", "metadata")
metadata_output_path = pathlib.Path.cwd().joinpath("metadata_v2")

# storage objects
connections = []
cam_objects = []
inst_components = []
expt_modalities = []
stim_epochs = []

### Parse procedures note files

In [ ]:
# -----------------------
# Label maps & regexes
# -----------------------

# Exact-label map for metadata table keys
KNOWN_FIELDS = {
    "Date / Experiment": "date_experiment",
    "Goal": "goal",
    "Overall impressions / conclusions": "overall_impressions",
    "Animal ID": "animal_id",
    "Animal state / anesthesia induction time": "anesthesia_induction",
    "Heating pad (state/temp)": "heating_pad",
    "Animal birthdate": "animal_birthdate",
    "Animal weight before": "weight_before",
    "Animal weight after": "weight_after",
    "Animal surgery history": "surgery_history",
    "Craniotomy coordinates": "craniotomy_coordinates",
    "Targettng Modifications": "targeting_modifications",  # keep label as-is in doc
    "Final probe depth (from pia)": "final_probe_depth_um",
    "Z(GNDscrew-REFscrew)": "z_gnd_screw_ref_screw",
    "Z(GNDprobe-REFprobe)": "z_gnd_probe_ref_probe",
    "Probe SN": "probe_sn",
    "Probe GND/REF leads connected to": "probe_gnd_ref_connected_to",
    "Probe orientation vs. AP axis": "probe_orientation_ap",
    "Number of probe channels": "probe_channels",
}

# Canonical map for resilient header matching
KNOWN_FIELDS_CANON = {
    "date experiment": "date_experiment",
    "goal": "goal",
    "overall impressions conclusions": "overall_impressions",
    "animal id": "animal_id",
    "animal state anesthesia induction time": "anesthesia_induction",
    "heating pad state temp": "heating_pad",
    "animal birthdate": "animal_birthdate",
    "animal weight before": "weight_before",
    "animal weight after": "weight_after",
    "animal surgery history": "surgery_history",
    "craniotomy coordinates": "craniotomy_coordinates",
    "targettng modifications": "targeting_modifications",
    "final probe depth from pia": "final_probe_depth_um",
    "z gndscrew refscrew": "z_gnd_screw_ref_screw",
    "z gndprobe refprobe": "z_gnd_probe_ref_probe",
    "probe sn": "probe_sn",
    "probe gnd ref leads connected to": "probe_gnd_ref_connected_to",
    "probe orientation vs ap axis": "probe_orientation_ap",
    "number of probe channels": "probe_channels",
}

# Notes headers (paragraphs or table rows, canonicalized)
NOTE_HEADERS_CANON = {
    "surgery notes": "surgery_notes",
    "metabond": "metabond_notes",
    "probe notes": "probe_notes",
    "penetration coverage notes": "penetration_notes",  # "penetration/coverage notes" -> canon removes '/'
    "implantation notes": "implantation_notes",
    "plug in 8 days after implant": "plugin_notes",  # "Plug-in" -> canon removes punctuation
}

# Regex patterns to capture inline note content (group 1 = trailing content)
NOTE_HEADER_PATTERNS = {
    "surgery_notes": re.compile(r"^\s*Surgery\s*notes\s*:?\s*(.*)$", re.IGNORECASE),
    "metabond_notes": re.compile(r"^\s*Metabond\s*:?\s*(.*)$", re.IGNORECASE),
    "probe_notes": re.compile(r"^\s*Probe\s*Notes\s*:?\s*(.*)$", re.IGNORECASE),
    "penetration_notes": re.compile(
        r"^\s*Penetration\s*(?:/|and\s+)?\s*coverage\s*notes\s*:?\s*(.*)$",
        re.IGNORECASE,
    ),
    "implantation_notes": re.compile(r"^\s*Implantation\s*notes\s*:?\s*(.*)$", re.IGNORECASE),
    "plugin_notes": re.compile(
        r"^\s*Plug[-\s]*in\s*8\s*days\s*after\s*implant\s*:?\s*(.*)$", re.IGNORECASE
    ),
}

# Date/experiment lines accept '/', en dash, or hyphen separator
DATE_LINE_RE = re.compile(r"^\s*(\d{1,2}/\d{1,2}/\d{2,4})\s*[/–-]\s*(.+?)\s*$")

# Shank depths lines; robust to "->" or HTML-like "-&gt;" and optional text between
SHANK_DEPTH_RE = re.compile(r"^shank\s*(\d+).*?(?:->|-&gt;)\s*([0-9?]+)", re.IGNORECASE)

# Numeric extractors
UM_NUM_RE = re.compile(r"(\d+)\s*(?:µm|um)?", re.IGNORECASE)
INT_RE = re.compile(r"\b(\d+)\b")

# Time extractor: accept colon or period, with optional AM/PM (case-insensitive)
# Examples matched: "1:16pm", "5.22", "13:20", "2:31 PM"
TIME_RE = re.compile(r"\b(\d{1,2}[:\.]\d{2})(?:\s*(am|pm))?\b", re.IGNORECASE)

# Specific regex for "Depth shank 4 entered relative to shank 1 ... 450[um]"
RELATIVE_SHANK4_RE = re.compile(
    r"Depth\s+shank\s*4\s+entered\s+relative\s+to\s+shank\s*1.*?(\d{2,5})\s*(?:um|µm)?",
    re.IGNORECASE,
)

# -----------------------
# Helpers
# -----------------------


def canon(s: str) -> str:
    """Lowercase, strip, remove punctuation and excessive spaces; normalize unicode."""
    if not s:
        return ""
    s = unicodedata.normalize("NFKC", s).strip().lower()
    s = re.sub(r"[,:/()µ\-]+", " ", s)  # remove punctuation-like chars (keeps words)
    s = re.sub(r"\s+", " ", s)  # collapse whitespace
    return s


def cell_text_robust(cell) -> str:
    """Return cell text, falling back to joining non-empty paragraphs if .text is empty."""
    txt = cell.text.strip()
    if txt:
        return txt
    parts = [p.text.strip() for p in cell.paragraphs if p.text.strip()]
    return "\n".join(parts)


def parse_value_multiline(text: str) -> List[str]:
    return [ln.strip() for ln in text.splitlines() if ln.strip()]


def parse_date_lines_from_text(block_text: str) -> List[Dict[str, str]]:
    items = []
    for ln in parse_value_multiline(block_text):
        m = DATE_LINE_RE.match(ln)
        if m:
            items.append({"date": m.group(1), "experiment": m.group(2).strip()})
    return items


def normalize_um(value: str) -> int | None:
    if not isinstance(value, str):
        return value
    m = UM_NUM_RE.search(value)
    return int(m.group(1)) if m else None


def normalize_int(value: str) -> int | None:
    if not isinstance(value, str):
        return value
    m = INT_RE.search(value)
    return int(m.group(1)) if m else None


def extract_time(text: str) -> str | None:
    """
    Extract time from text supporting colon or period separator and optional AM/PM.
    IMPORTANT: Returns the matched time string EXACTLY as it appears (no normalization).
    Examples returned: '1:16pm', '5.22', '2:31 PM'
    """
    m = TIME_RE.search(text)
    if not m:
        return None
    # Return the exact matched substring (includes AM/PM if present)
    return m.group(0)


def is_date_experiment_header(kcanon: str) -> bool:
    """Detect 'Date / Experiment' headers even with extra text like '(delete irrelevant fields)'."""
    if "date experiment" in kcanon:
        return True
    return "date" in kcanon and "experiment" in kcanon


def is_known_field_start(kcanon: str) -> bool:
    """Any canonical key that maps to a known field (excluding date_experiment) ends the date block."""
    return kcanon in KNOWN_FIELDS_CANON and KNOWN_FIELDS_CANON[kcanon] != "date_experiment"


def resolve_note_key_from_canon(kcanon: str) -> str | None:
    """Return notes data key if this canonical header represents a notes section."""
    if kcanon in NOTE_HEADERS_CANON:
        return NOTE_HEADERS_CANON[kcanon]
    for note_hdr_canon, data_key in NOTE_HEADERS_CANON.items():
        if kcanon.startswith(note_hdr_canon) or note_hdr_canon in kcanon:
            return data_key
    return None


def strip_note_header(line: str) -> str:
    """
    Remove a note header prefix from the given line and return trailing content only.
    If no header is present, returns the line unchanged.
    """
    for pat in NOTE_HEADER_PATTERNS.values():
        m = pat.match(line)
        if m:
            return m.group(1).strip()
    return line


def append_note_lines(data: Dict[str, Any], key: str, lines: List[str]) -> None:
    """
    Append lines to a notes field without duplicates and stripping any header prefixes.
    Preserves order of first occurrence.
    """
    existing_lines = [ln.strip() for ln in data.get(key, "").splitlines() if ln.strip()]
    seen = set(existing_lines)
    for ln in lines:
        cleaned = strip_note_header(ln).strip()
        if not cleaned:
            continue
        if cleaned not in seen:
            existing_lines.append(cleaned)
            seen.add(cleaned)
    data[key] = "\n".join(existing_lines)


# -----------------------
# Parsers
# -----------------------


def parse_metadata_table(
    doc: Document, table_index: int = 0
) -> Tuple[Dict[str, Any], pd.DataFrame]:
    """Parse the main metadata table, including a multi-row Date/Experiment block and notes-in-table."""
    table = doc.tables[table_index]

    data: Dict[str, Any] = {
        "date_experiment": [],
        "goal": "",
        "overall_impressions": "",
        "animal_id": None,
        "anesthesia_induction": "",
        "heating_pad": "",
        "animal_birthdate": "",
        "weight_before": "",
        "weight_after": "",
        "surgery_history": "",
        "craniotomy_coordinates": "",
        "targeting_modifications": [],
        "final_probe_depth_um": None,
        "z_gnd_screw_ref_screw": "",
        "z_gnd_probe_ref_probe": "",
        "probe_sn": "",
        "probe_gnd_ref_connected_to": "",
        "probe_orientation_ap": "",
        "probe_channels": None,
        # Paragraph-only fields will be filled later (initialized here for unified dict)
        "entered_brain_time": None,
        "depth_shank4_relative_um": None,
        "entered_dead_zone": {},
        "entered_piriform": {},
        "final_depth_um": None,
        "hit_final_depth_time": None,
        "metabond_applied_time": None,
        "relyx_started_time": None,
        "surgery_end_time": None,
        "surgery_notes": "",
        "metabond_notes": "",
        "probe_notes": "",
        "penetration_notes": "",
        "implantation_notes": "",
        "plugin_notes": "",
    }

    in_date_block = False

    for row in table.rows:
        cells = row.cells
        if len(cells) < 1:
            continue

        raw_key = cell_text_robust(cells[0])
        key_canon = canon(raw_key)
        raw_val = "\n".join(cell_text_robust(c) for c in cells[1:]).strip()

        # Date/Experiment block collection across subsequent rows (table-only)
        if in_date_block:
            if is_known_field_start(key_canon) or resolve_note_key_from_canon(key_canon):
                in_date_block = False
            else:
                combined = [cell_text_robust(cells[0])]
                if len(cells) > 1:
                    combined.extend(cell_text_robust(c) for c in cells[1:])
                items = parse_date_lines_from_text("\n".join(t for t in combined if t.strip()))
                for it in items:
                    if it not in data["date_experiment"]:
                        data["date_experiment"].append(it)
                continue

        # Skip pure placeholder unless it's part of Date/Experiment header
        if "delete irrelevant" in key_canon and not is_date_experiment_header(key_canon):
            note_key = resolve_note_key_from_canon(key_canon)
            if note_key:
                append_note_lines(data, note_key, parse_value_multiline(raw_val))
            continue

        # Start Date/Experiment header
        if is_date_experiment_header(key_canon):
            if raw_val:
                data["date_experiment"].extend(parse_date_lines_from_text(raw_val))
            in_date_block = True
            continue

        # Notes in table rows
        note_key = resolve_note_key_from_canon(key_canon)
        if note_key:
            append_note_lines(data, note_key, parse_value_multiline(raw_val))
            continue

        # Normal known-field parsing
        if raw_key in KNOWN_FIELDS:
            field = KNOWN_FIELDS[raw_key]

            if field == "targeting_modifications":
                data[field] = parse_value_multiline(raw_val)
            elif field == "final_probe_depth_um":
                data[field] = normalize_um(raw_val)
            elif field == "probe_channels":
                data[field] = normalize_int(raw_val)
            elif field == "animal_id":
                id_match = re.search(r"\b(\d{4,})\b", raw_val) or re.search(
                    r"\b(\d{4,})\b", raw_key
                )
                data[field] = id_match.group(1) if id_match else (raw_val or raw_key)
            else:
                data[field] = "\n".join(parse_value_multiline(raw_val))
        else:
            # Canonical fallback (excluding date_experiment)
            if (
                key_canon in KNOWN_FIELDS_CANON
                and KNOWN_FIELDS_CANON[key_canon] != "date_experiment"
            ):
                field = KNOWN_FIELDS_CANON[key_canon]
                if field == "targeting_modifications":
                    data[field] = parse_value_multiline(raw_val)
                elif field == "final_probe_depth_um":
                    data[field] = normalize_um(raw_val)
                elif field == "probe_channels":
                    data[field] = normalize_int(raw_val)
                elif field == "animal_id":
                    id_match = re.search(r"\b(\d{4,})\b", raw_val) or re.search(
                        r"\b(\d{4,})\b", raw_key
                    )
                    data[field] = id_match.group(1) if id_match else (raw_val or raw_key)
                else:
                    data[field] = "\n".join(parse_value_multiline(raw_val))
            # else: ignore unknown keys for schema cleanliness

    date_expt_df = (
        pd.DataFrame(data["date_experiment"])
        if data["date_experiment"]
        else pd.DataFrame(columns=["date", "experiment"])
    )
    return data, date_expt_df


def parse_paragraph_sections(doc: Document, data: Dict[str, Any]) -> None:
    """
    Parse times, depths, and free-text notes from paragraphs (NOT from tables),
    iterating per line to prevent duplicates and capture inline header content cleanly.

    Key behavior:
     - Match phrases on raw lines and extract times via TIME_RE (supports '.' and AM/PM).
     - Use a specific regex for 'Depth shank 4 entered relative...' to capture '450' even without 'um'.
     - Accept 'Never' and question-mark depths in shank lines.
     - Do NOT normalize times; return exactly as matched (e.g., '1:16pm', '5.22').
    """
    current_notes_key = None
    in_dead_zone = False
    in_piriform = False

    for para in doc.paragraphs:
        # Iterate per physical line inside the paragraph to separate mixed content
        for raw_line in para.text.splitlines():
            line = raw_line.strip()
            if not line:
                continue

            # Normalize common artifacts before matching (HTML-like "->")
            line_norm = line.replace("-&gt;", "->").replace("→", "->")
            lc = canon(line_norm)  # for header detection only

            # End notes block when K/X anesthesia section starts
            if lc.startswith("k x anesthesia"):
                current_notes_key = None
                in_dead_zone = False
                in_piriform = False
                continue

            # Notes header with inline content? Capture header + trailing content immediately
            appended_inline = False
            for note_key, pat in NOTE_HEADER_PATTERNS.items():
                m = pat.match(line)  # use raw line to preserve punctuation and spacing
                if m:
                    current_notes_key = note_key
                    trailing = m.group(1).strip()
                    if trailing:
                        append_note_lines(data, current_notes_key, [trailing])
                        appended_inline = True
                    break
            if not appended_inline:
                # Fallback notes header detection (canonical startswith/contains)
                nk = resolve_note_key_from_canon(lc)
                if nk and nk != current_notes_key:
                    current_notes_key = nk
                    in_dead_zone = False
                    in_piriform = False
                    # header line without inline content; proceed

            # Dead zone / Piriform headers (end notes)
            if "(for apc) entered dead zone" in lc:
                in_dead_zone = True
                in_piriform = False
                current_notes_key = None
                continue

            if "(for apc) entered piriform" in lc:
                in_piriform = True
                in_dead_zone = False
                current_notes_key = None
                continue

            # Collect shank depths when in a zone section (paragraph-only)
            if in_dead_zone or in_piriform:
                # Accept 'Never' and values ending with '?'
                m = SHANK_DEPTH_RE.match(line_norm)
                if m:
                    shank = int(m.group(1))
                    val = m.group(2)
                    if isinstance(val, str) and val.strip().lower() == "never":
                        depth = None
                    else:
                        # strip any non-digits like '?'
                        digits = re.search(r"(\d+)", val)
                        depth = int(digits.group(1)) if digits else None
                    bucket = "entered_dead_zone" if in_dead_zone else "entered_piriform"
                    data[bucket][shank] = depth
                continue

            # Times & scalar fields (paragraph-only) — match PHRASES on raw line, extract time with TIME_RE
            if "Entered brain at" in line_norm:
                t = extract_time(line_norm)
                if t:
                    data["entered_brain_time"] = t

            m_rel = RELATIVE_SHANK4_RE.search(line_norm)
            if m_rel:
                data["depth_shank4_relative_um"] = int(m_rel.group(1))

            if line_norm.upper().startswith("FINAL DEPTH"):
                m = re.search(r"(\d+)", line_norm)
                if m:
                    data["final_depth_um"] = int(m.group(1))

            if "Hit final depth at" in line_norm:
                t = extract_time(line_norm)
                if t:
                    data["hit_final_depth_time"] = t

            if "Metabond applied at" in line_norm or "Bruno cement applied at" in line_norm:
                t = extract_time(line_norm)
                if t:
                    data["metabond_applied_time"] = t

            if "RelyX started at" in line_norm:
                t = extract_time(line_norm)
                if t:
                    data["relyx_started_time"] = t

            if "Surgery ended at" in line_norm:
                t = extract_time(line_norm)
                if t:
                    data["surgery_end_time"] = t

            # Append the line to the active note (if any), once and header-stripped.
            if current_notes_key:
                if not appended_inline:
                    append_note_lines(data, current_notes_key, [line])
                continue
            # else: ignore non-note lines that don't belong to recognized sections


def parse_kx_anesthesia_table(doc: Document) -> pd.DataFrame:
    """
    Parse the K/X anesthesia table into a DataFrame.
    Robust to rows with missing columns and note-only lines.
    """
    kx_rows: List[Dict[str, Any]] = []

    def is_kx_header_row(headers: List[str]) -> bool:
        hs = [h.strip().lower() for h in headers]
        return "time" in hs and "agent" in hs and "concentration" in hs

    # Find a table whose first row looks like the K/X header
    for table in doc.tables:
        if len(table.rows) == 0:
            continue

        header_cells = [c.text.strip() for c in table.rows[0].cells]
        if not header_cells:
            continue

        if is_kx_header_row(header_cells):
            # Parse table rows
            last_row_dict = None
            for r_idx, row in enumerate(table.rows[1:], start=1):
                cells = [c.text.strip() for c in row.cells]
                row_dict: Dict[str, Any] = {}
                if len(cells) >= 1:
                    row_dict["time"] = cells[0] or None
                if len(cells) >= 2:
                    row_dict["agent"] = cells[1] or None
                if len(cells) >= 3:
                    row_dict["concentration"] = cells[2] or None
                if len(cells) >= 4:
                    row_dict["volume_ul"] = cells[3] or None
                if len(cells) >= 5:
                    row_dict["reason"] = cells[4] or None

                # Note-only lines (single cell after a data row)
                if len(cells) == 1 and cells[0] and last_row_dict:
                    if not last_row_dict.get("reason"):
                        last_row_dict["reason"] = cells[0]
                    else:
                        kx_rows.append(
                            {
                                "time": None,
                                "agent": None,
                                "concentration": None,
                                "volume_ul": None,
                                "reason": cells[0],
                            }
                        )
                    continue

                # Skip truly empty rows
                if not any(cells):
                    continue

                kx_rows.append(row_dict)
                last_row_dict = row_dict
            break  # stop after the first matching table

    kx_df = (
        pd.DataFrame(kx_rows)
        if kx_rows
        else pd.DataFrame(columns=["time", "agent", "concentration", "volume_ul", "reason"])
    )
    return kx_df


# -----------------------
# Orchestrator
# -----------------------


def parse_docx_full(
    path: str, metadata_table_index: int = 0
) -> Tuple[Dict[str, Any], pd.DataFrame, pd.DataFrame]:
    """
    End-to-end parser:
     - Parse metadata table (incl. Date/Experiment block; table-only, plus notes-in-table)
     - Parse paragraphs: times, depths, notes (paragraph-only, line-level with inline header capture; NO time normalization)
     - Parse K/X anesthesia table into a DataFrame
    Returns: (data, date_expt_df, kx_df)
    """
    doc = Document(path)
    if len(doc.tables) == 0:
        raise ValueError("No tables found in the document.")

    data, date_expt_df = parse_metadata_table(doc, table_index=metadata_table_index)
    parse_paragraph_sections(doc, data)
    kx_df = parse_kx_anesthesia_table(doc)

    # (Optional) unify final depth fields when table depth missing
    if data.get("final_probe_depth_um") is None and data.get("final_depth_um") is not None:
        data["final_probe_depth_um"] = data["final_depth_um"]

    return data, date_expt_df, kx_df


if __name__ == "__main__":
    # Change this to your file path
    parsed_data, date_expt_df, kx_df = parse_docx_full(subject_notes)

    # --- Summary ---
    print("=== Structured Fields ===")
    summary_keys = [
        "animal_id",
        "goal",
        "overall_impressions",
        "anesthesia_induction",
        "heating_pad",
        "animal_birthdate",
        "weight_before",
        "weight_after",
        "surgery_history",
        "craniotomy_coordinates",
        "targeting_modifications",
        "final_probe_depth_um",
        "probe_sn",
        "probe_gnd_ref_connected_to",
        "probe_orientation_ap",
        "probe_channels",
        # paragraph-only fields
        "entered_brain_time",
        "depth_shank4_relative_um",
        "final_depth_um",
        "hit_final_depth_time",
        "metabond_applied_time",
        "relyx_started_time",
        "surgery_end_time",
        # notes (from table and/or paragraphs)
        "surgery_notes",
        "metabond_notes",
        "probe_notes",
        "penetration_notes",
        "implantation_notes",
        "plugin_notes",
    ]
    for k in summary_keys:
        if k in [
            "surgery_notes",
            "metabond_notes",
            "probe_notes",
            "penetration_notes",
            "implantation_notes",
            "plugin_notes",
        ]:
            DASH_CLASS = r"[\u2013\u2014\-]"  # en dash, em dash, hyphen
            # Remove non-printable/control characters (anything not JSON-safe)
            parsed_data[k] = re.sub(DASH_CLASS + r"+", " ", parsed_data.get(k))
            parsed_data[k] = re.sub(r"[^\x20-\x7E\u00A0-\uFFFF]", "", parsed_data.get(k))
        print(f"{k}: {parsed_data.get(k)}")

    print("\n=== Date / Experiment ===")
    print(date_expt_df)

    print("\n=== Dead Zone (µm) ===", parsed_data["entered_dead_zone"])
    print("=== Piriform (µm) ===", parsed_data["entered_piriform"])

    print("\n=== K/X Anesthesia Table ===")
    print(kx_df)

    probe_serial_number = parsed_data.get("probe_sn")  # get from procedures

=== Structured Fields ===
animal_id: 815154
goal: NP2.0 implantation in aPC (elbow)
overall_impressions: Good targeting, excellent signals.
anesthesia_induction: Craniotomy:  isoflurane  11:57
Implantation: K/X / isoflurane /
heating_pad: ON / 41.0
animal_birthdate: 04/22/2025
weight_before: Craniotomy: 23.9 g
Implantation: 25.7 g
weight_after: Craniotomy: 24.1 g
Implantation: 31.1 g
surgery_history: HP attachment: 07/01/2025
Thermistor:    n/a
craniotomy_coordinates: 1,150 µm PRCS, shank 1: 1,575 µm L + shank 4: 2,325 µm L
targeting_modifications: []
final_probe_depth_um: 3745
probe_sn: 23107807252
probe_gnd_ref_connected_to: GND: cerebellum screw / REF: cerebellum screw / Shorted
probe_orientation_ap: Sites facing head
probe_channels: 384
entered_brain_time: 11:35
depth_shank4_relative_um: 450
final_depth_um: 3745
hit_final_depth_time: 12:19
metabond_applied_time: 12:37
relyx_started_time: 1:20
surgery_end_time: 4:54pm
surgery_notes: good craniotomy and durotomy (a little dura in the

### Pirouette Instrument

In [ ]:
if "pirouette" in current_experiment:
    # load parameters from json configs
    pirouette_metadata_path = pathlib.Path.cwd().joinpath(
        metadata_path, "AindBehaviorPirouetteRig.json"
    )

    # Open and read JSON file
    with open(pirouette_metadata_path, "r", encoding="utf-8") as file:
        pirouette_json = json.load(file)

    # start dynamically mapping
    computer = Computer(name=pirouette_json["rig_name"])
    cameras = pirouette_json["camera_controller"]["cameras"]
    camera_names = list(cameras.keys())
    camera_ids = [cameras[name]["serial_number"] for name in camera_names]

    # probe info
    probe_metadata_path = dataset_root.joinpath("ecephys", "probe.json")

    with open(probe_metadata_path, "r", encoding="utf-8") as file:
        probe_json = json.load(file)

    probe_model_info = (
        probe_json["probes"][0]["annotations"]["name"]
        .replace(" - ", " (")
        .replace("Multishank", "Multi Shank")
        + ")"
    )

    # Instrument Metadata
    """HARP"""
    digitial_out = DAQChannel(channel_name="OUT_1", channel_type="Digital Output")

    clock_input = DAQChannel(channel_name="CLK_IN", channel_type="Digital Input")

    analog_expansion_input = DAQChannel(
        channel_name="Expansion",
        channel_type="Analog Input",
    )

    output_expander_channels = [digitial_out, clock_input, analog_expansion_input]
    harp_expander = HarpDevice(
        name="Harp Output Expander",
        harp_device_type=HarpDeviceType.OUTPUTEXPANDER,
        core_version="1.2",
        channels=output_expander_channels,
        is_clock_generator=False,
    )

    harp_sync = HarpDevice(
        name="Harp White Rabbit",
        harp_device_type=HarpDeviceType.CLOCKSYNCHRONIZER,
        core_version="1.1",
        is_clock_generator=True,
    )

    """EPHYS"""
    port1 = ProbePort(index=1, probes=[probe_id])

    headstage = Device(name="ONIX Headstage Neuropixels 2.0e")

    probe1 = EphysProbe(
        name=probe_id,
        serial_number=probe_serial_number,
        probe_model=probe_model_info,
        headstage=headstage,
    )

    onix_board = OpenEphysAcquisitionBoard(
        name="Onix Breakout Board",
        firmware_version="1.3",
        ports=[port1],
    )

    # Manipulator is the one used during surgery
    ephys_assembly = EphysAssembly(
        name="Chronic_ephys_assembly",
        manipulator=Manipulator(
            name="Manipulator scientifica",
            manufacturer=Organization.OTHER,
            notes="Manipulator is only used during the chronic implantation procedure.",
        ),
        probes=[probe1],
    )

    """Commutator"""
    commutator = Device(
        name="Coaxial Commutator",
        manufacturer=Organization.OEPS,
    )

    """Hall Effect Magnetic Encoder"""
    mag_encoder = Device(
        name="Magnetic Encoder",
        serial_number="U1 AS504BA",
        manufacturer=Organization.AIND,
        notes="encodes the relative rotation of a ring magnet (K&J Magnetics: 1/4 inch x 1/8 inch x 1/4 inch, ID: N42 R424DIA, DIA Magnetized) glued onto the coaxial tether",
    )

    """Cameras"""
    # camera lens
    lens = Lens(
        name="Camera lens",
        manufacturer=Organization.TAMRON,
        notes="Tamron 12VG412ASIR lens.",
    )

    for cam_name in camera_names:
        cam_objects.append(
            Camera(
                name=cam_name,
                detector_type="Camera",
                data_interface="USB",
                manufacturer=Organization.FLIR,
                frame_rate=pirouette_json["camera_controller"]["frame_rate"],
                frame_rate_unit=FrequencyUnit.HZ,
                sensor_width=1440,
                sensor_height=1080,
                chroma="Monochrome",
                gain=cameras[cam_name]["gain"],
                recording_software=Software(
                    name="Spinnaker SDK",
                    version="1.29.0.5",
                ),
                notes="Model: Blackfly S BFS-U3-04S2M-CS",
            )
        )

    cam_assemblies = []
    for cam_obj in cam_objects:
        if "Top" in cam_obj.name:
            rel_pos = AnatomicalRelative.SUPERIOR
        else:
            rel_pos = AnatomicalRelative.LEFT
        cam_assemblies.append(
            CameraAssembly(
                name=f"{cam_obj.name}_assembly",
                target=CameraTarget.BODY,
                relative_position=[rel_pos],
                camera=cam_obj,
                lens=lens,
            )
        )

    """Connections"""
    # camera triggers
    for cam_name in camera_names:
        connections.append(
            Connection(
                source_device="Harp Output Expander",
                source_port="OUT_1",
                target_device=cam_name,
            )
        )

        connections.append(
            Connection(
                source_device=cam_name,
                target_device=computer.name,
            )
        )

    # inputs into harp output expander
    connections.append(
        Connection(
            source_device="Magnetic Encoder",
            target_device="Harp Output Expander",
            target_port="Expansion",
        )
    )

    connections.append(
        Connection(
            source_device="Harp White Rabbit",
            target_device="Harp Output Expander",
            target_port="CLK_IN",
        )
    )

    # Ephys connections
    connections.append(
        Connection(
            source_device="Onix Breakout Board",
            target_device=computer.name,
        )
    )

    # Commutator connections
    connections.append(
        Connection(
            source_device="Coaxial Commutator",
            target_device="Onix Breakout Board",
        )
    )

    connections.append(
        Connection(
            source_device="Coaxial Commutator",
            target_device=computer.name,
        )
    )

    # magnetic encoder connections
    connections.append(
        Connection(
            source_device="Magnetic Encoder",
            target_device="Harp Output Expander",
            target_port="Expansion",
        )
    )

    """Pirouette Box"""
    beh_box = Enclosure(
        name="Pirouette Behavior Box",
        size=Scale(scale=[8.0, 15.0, 32.0]),
        size_unit=SizeUnit.IN,
        internal_material="Bedding and nesting material",
        external_material="Acrylic",
        grounded=True,
        laser_interlock=False,
        air_filtration=True,
    )

    """IR Illumination of the Behavior Box"""
    # Lens for defracting light
    IR_lens = Lens(
        name="IR Convex Lens",
        manufacturer=Organization.THORLABS,
    )

    # IR light
    IR_illumination = LightEmittingDiode(
        name="IR Light Source",
        manufacturer=Organization.THORLABS,
        wavelength=810,
        wavelength_unit=SizeUnit.NM,
    )

    # Compile instrument components
    pirouette_components = [
        computer,
        harp_expander,
        harp_sync,
        ephys_assembly,
        commutator,
        mag_encoder,
        beh_box,
        IR_illumination,
        IR_lens,
        onix_board,
    ]
    for component in pirouette_components:
        inst_components.append(component)
    for cam_assembly in cam_assemblies:
        inst_components.append(cam_assembly)

    # modalities
    expt_modalities.append(Modality.ECEPHYS)
    expt_modalities.append(Modality.BEHAVIOR_VIDEOS)


### Delphi Instrument

In [ ]:
if "delphi" in current_experiment:
    # load .jsonl file
    metadatafiles = os.listdir(metadata_path)
    for file in metadatafiles:
        if "HardwareSettings" in file:
            delphi_metadata_path = pathlib.Path.cwd().joinpath(metadata_path, file)

            # Open the JSONL file
            with open(delphi_metadata_path, "r") as jsonl_file:
                # Read and parse each line into a list of dictionaries
                delphi_hardware = [json.loads(line) for line in jsonl_file]

        if "RuleSettings" in file:
            rule_metadata_path = pathlib.Path.cwd().joinpath(metadata_path, file)

            # Open the JSONL file
            with open(rule_metadata_path, "r") as jsonl_file:
                # Read and parse each line into a list of dictionaries
                delphi_rules = [json.loads(line) for line in jsonl_file]

    """Computer"""
    if "pirouette" not in current_experiment:
        delphi_computer = Computer(name=delphi_computer_id)
        inst_components.append(delphi_computer)
    else:
        delphi_computer = Computer(name=pirouette_json["rig_name"])  # Same as pirouette computer

    """Delphi Controller"""
    # channels
    odor_transitions = delphi_rules[0]["value"]["rule"]["stateDefinitions"]
    odor_channels = []
    odor_names = []
    for odor in odor_transitions:
        odor_name = odor["name"]
        odor_idx = int(math.log2(odor["odorIndex"]))
        if odor_name != "DefaultState":
            odor_names.append(odor_name)
            odor_channels.append(
                OlfactometerChannel(
                    channel_index=odor_idx,
                    channel_type=OlfactometerChannelType.ODOR,
                    flow_unit="mL/min",
                )
            )

    # delphi controller device
    harp_delphi_controller = Olfactometer(
        manufacturer=Organization.AIND,
        name="Delphi Controller",
        harp_device_type=HarpDeviceType.OLFACTOMETER,
        core_version="1.0",
        channels=odor_channels,
        is_clock_generator=False,
        notes="whoami=1409 and flow rate is actually set to 75 mL/min",
    )
    inst_components.append(harp_delphi_controller)

    connections.append(
        Connection(
            source_device="Delphi Controller",
            target_device=delphi_computer.name,
        )
    )

    """Poke Port"""
    poke_port = Device(
        name="Poke Port",
        manufacturer=Organization.AIND,
        notes="https://github.com/AllenNeuralDynamics/harp.peripheral.poke-port",
    )
    inst_components.append(poke_port)

    connections.append(
        Connection(
            source_device="Poke Port",
            target_device="Delphi Controller",
            target_port=f"POKE0_Pin {delphi_hardware[0]['value']['delphiController']['pokePin']}",
        )
    )

    # Pirouette doesn't use cameras from delphi workflow, but delphi only does
    if "pirouette" not in current_experiment:
        """Cameras"""
        # camera lens
        lens = Lens(
            name="Camera lens",
            manufacturer=Organization.COMPUTAR,
            notes="Computar M3Z1228C-MP lens.",
        )

        delphi_cameras = delphi_hardware[0]["value"]["cameraSettings"]

        for cam in [delphi_cameras]:
            cam_objects.append(
                Camera(
                    name=cam["cameraName"],
                    detector_type="Camera",
                    data_interface="USB",
                    manufacturer=Organization.FLIR,
                    frame_rate=cam["frameRate"],
                    frame_rate_unit=FrequencyUnit.HZ,
                    sensor_width=1440,
                    sensor_height=1080,
                    chroma="Monochrome",
                    recording_software=Software(
                        name="Spinnaker SDK",
                        version="1.29.0.5",
                    ),
                    notes="Model: Blackfly S BFS-U3-04S2M-CS",
                )
            )
            # camera connection
            connections.append(
                Connection(
                    source_device="Delphi Controller",
                    source_port="CAM0",
                    target_device=cam["cameraName"],
                )
            )

            connections.append(
                Connection(
                    source_device=cam["cameraName"],
                    target_device=delphi_computer.name,
                )
            )

        cam_assemblies = []
        for cam_obj in cam_objects:
            cam_assemblies.append(
                CameraAssembly(
                    name=f"{cam_obj.name}_assembly",
                    target=CameraTarget.BODY,
                    relative_position=[AnatomicalRelative.SUPERIOR],
                    camera=cam_obj,
                    lens=lens,
                )
            )

        # Add to instrument components
        for cam_assembly in cam_assemblies:
            inst_components.append(cam_assembly)

        """Delphi Cage"""
        delphi_cage = Enclosure(
            name="Delphi Behavior Cage",
            size=Scale(scale=[14.5, 7.0, 7.0]),
            size_unit=SizeUnit.IN,
            internal_material="Bedding and nesting material and a hut",
            external_material="Acrylic",
            grounded=False,
            laser_interlock=False,
            air_filtration=True,
            notes="https://www.allentowninc.com/rodent-housing/nexgen/",
        )
        inst_components.append(delphi_cage)

        # modalities
        if Modality.BEHAVIOR_VIDEOS not in expt_modalities:
            expt_modalities.append(Modality.BEHAVIOR_VIDEOS)
        # expt_modalities.append(Modality.BEHAVIOR) # Requires lickspout which is not apart of the Delphi setup


In [49]:
# Build the Instrument Object
inst = Instrument(
    location=experiment_room,
    instrument_id=instrument,
    modification_date=date(today.year, today.month, today.day),
    modalities=expt_modalities,
    coordinate_system=CoordinateSystemLibrary.ARENA_RBT,  # Double check that this is correct
    components=inst_components,
    connections=connections,
)

In [50]:
# write to json
if __name__ == "__main__":
    serialized = inst.model_dump_json()
    deserialized = Instrument.model_validate_json(serialized)
    deserialized.write_standard_file(output_directory=metadata_output_path)

### Acquisition JSON

In [51]:
# Import libraries
from aind_data_schema.components.identifiers import Software, Code
from aind_data_schema.core.acquisition import (
    Acquisition,
    AcquisitionSubjectDetails,
    DataStream,
    StimulusEpoch,
)
from aind_data_schema.components.configs import (
    ManipulatorConfig,
    EphysAssemblyConfig,
    ProbeConfig,
    OlfactometerConfig,
    OlfactometerChannelInfo,
)
from aind_data_schema.components.coordinates import (
    Translation,
    AtlasCoordinate,
    AtlasLibrary,
    CoordinateSystemLibrary,
)

from aind_data_schema_models.brain_atlas import CCFv3
from aind_data_schema_models.stimulus_modality import StimulusModality

In [ ]:
"""Universal Session Info"""

# Acquistion software
spinview = Software(
    name="Spinnaker SDK",
    version="1.29.0.5",
)

bonsai = Software(name="Bonasi", version="2.9")

"""Pirouette Session"""
if "pirouette" in current_experiment:
    # Open Ephys GUI
    opes_gui = Software(
        name="Open Ephys GUI",
        version="1.0.1 - https://open-ephys.org/gui/",
    )

    # Pirouette Bonsai Acquisition Code
    pirouette_bonsai_acq = Code(
        name="Pirouette Bonsai Acquisition",
        url="https://github.com/AllenNeuralDynamics/Aind.Behavior.Pirouette/src/WrappedPirouette.bonsai",
        core_dependency=bonsai,
    )

    # LifeAlert
    lifealert = Code(
        name="LifeAlert",
        url="https://github.com/AllenNeuralDynamics/lifealert",
        language="Python",
    )

    # Ephys Assembly
    probe_config = ProbeConfig(
        primary_targeted_structure=CCFv3.PIR,
        device_name=probe_id,
        atlas_coordinate=AtlasCoordinate(
            coordinate_system=AtlasLibrary.CCFv3_10um,
            translation=[6854, 5759, 2114],
        ),
        coordinate_system=CoordinateSystemLibrary.BREGMA_ARI,
        transform=[],
        notes=("Probe is chronically implanted so manipulator is only used during implantation."),
    )
    ephys_assembly_config = EphysAssemblyConfig(
        device_name=ephys_assembly.name,
        manipulator=ManipulatorConfig(
            device_name="scientifica manipulator",
            coordinate_system=CoordinateSystemLibrary.BREGMA_ARI,
            local_axis_positions=Translation(
                translation=[0, 0, 3971],  # dynamically map from procedures
            ),
        ),
        probes=[probe_config],
    )

"""Delphi Session"""
if "delphi" in current_experiment:
    # Delphi Bonsai Acquisition Software
    delphi_bonsai_acq = Code(
        name="Delphi Bonsai Acquisition",
        url="https://github.com/goatsofnaxos/Delphi/src/DelphiMain.bonsai",
        core_dependency=bonsai,
    )

    # Olfactometer Config
    odor_channel_info = []
    for i, odor in enumerate(odor_names):
        odor_channel_info.append(
            OlfactometerChannelInfo(
                channel_index=odor_channels[i].channel_index,
                odorant=odor,
                dilution=0.0,
            )
        )

    delphi_controller_config = OlfactometerConfig(
        device_name="Delphi Controller",
        channel_configs=odor_channel_info,
    )


In [24]:
# Acquisition information mapping
if "pirouette" in current_experiment:
    # load parameters from json configs
    pirouette_metadata_path = pathlib.Path.cwd().joinpath(
        metadata_path, "AindBehaviorSessionModel.json"
    )

    # Open and read JSON file
    with open(pirouette_metadata_path, "r", encoding="utf-8") as file:
        pirouette_session_json = json.load(file)

    # Use metadata file creation time to mark the start of the experiment
    creation_time = os.path.getctime(pirouette_metadata_path)
    expt_start_time = datetime.fromtimestamp(creation_time, tz=timezone.utc)

    try:
        int(pirouette_session_json["subject"])
        subject_id = pirouette_session_json["subject"]
    except ValueError:
        # Manual input of subject id
        subject_id = subject

    experimenters = pirouette_session_json["experimenter"]

"""DELPHI ONLY REQUIRES MANUAL INPUT OF SESSION METADATA UNTIL REFACTOR"""
if ("delphi" in current_experiment) and ("pirouette" not in current_experiment):
    # Manual input of session metadata
    subject_id = subject

    # Use metadata file creation time to mark the start of the experiment
    creation_time = os.path.getctime(delphi_metadata_path)
    expt_start_time = datetime.fromtimestamp(creation_time, tz=timezone.utc)

In [25]:
"""Find the most recently created file in a directory"""


def get_creation_timestamp(p: pathlib.Path) -> float | None:
    """
    Return the file's creation timestamp in seconds since epoch if available.
    - Windows: st_ctime (creation)
    - macOS: st_birthtime (creation)
    - Linux: creation time not generally available -> return None
    """
    st = p.stat()
    # macOS birth time
    if hasattr(st, "st_birthtime"):
        return st.st_birthtime
    # Windows creation time
    if os.name == "nt":
        return st.st_ctime
    # Linux/Unix: typically no birth time
    return None


def most_recent_created_file(dir_path: str, include_subdirs: bool = False):
    """
    Find the file with the most recent *creation* time in the given directory.
    If creation time is unavailable (e.g., Linux), falls back to modification time.

    Returns:
        (Path or None, datetime (UTC) or None, bool used_fallback)
    """
    root = pathlib.Path(dir_path)
    if not root.is_dir():
        raise NotADirectoryError(f"Not a directory: {dir_path}")

    iterator = root.rglob("*") if include_subdirs else root.glob("*")

    latest_path = None
    latest_ts = None
    used_fallback = False

    for p in iterator:
        if not p.is_file():
            continue

        cts = get_creation_timestamp(p)
        if cts is not None:
            ts = cts
        else:
            # Fallback for platforms without true creation time
            ts = p.stat().st_mtime
            used_fallback = True

        if latest_ts is None or ts > latest_ts:
            latest_ts = ts
            latest_path = p

    if latest_path is None:
        return None, None, False

    dt_utc = datetime.fromtimestamp(latest_ts, tz=timezone.utc)
    return latest_path, dt_utc, used_fallback


In [27]:
""" Pool until keypress to get end time of acquistion
    NOTE: Other actions can be performed while pooling like QC checks on fixed intervals
"""
user32 = ctypes.windll.user32


def pressed(vk):
    # returns True if key is down now
    return user32.GetAsyncKeyState(vk) & 0x8000 != 0


# Virtual-key codes
VK_CONTROL = 0x11
VK_SHIFT = 0x10
VK_Q = 0x51

print("Running. Press Ctrl+Shift+Q to stop...")
while True:
    time.sleep(0.05)
    if pressed(VK_CONTROL) and pressed(VK_SHIFT) and pressed(VK_Q):
        print("Ctrl+Shift+Q detected. Stopping and getting aquistion end time.")
        path, dt_utc, fallback = most_recent_created_file(
            dataset_root.joinpath("behavior-videos"), include_subdirs=True
        )
        expt_end_time = dt_utc
        break

print("Stopped.")

Running. Press Ctrl+Shift+Q to stop...
Ctrl+Shift+Q detected. Stopping and getting aquistion end time.
Stopped.


In [28]:
# Stimulus Epochs
acq_code = []
acq_device_list = []
device_configs = []
if "delphi" in current_experiment:
    delphi_stim = StimulusEpoch(
        stimulus_name="Odor Delivery",
        stimulus_modalities=[
            StimulusModality.OLFACTORY,
            StimulusModality.FREE_MOVING,
        ],
        stimulus_start_time=expt_start_time,
        stimulus_end_time=expt_end_time,
        configurations=[delphi_controller_config],
        code=delphi_bonsai_acq,
        active_devices=["Delphi Controller"],
    )

    stim_epochs.append(delphi_stim)
    acq_code.append(delphi_bonsai_acq)
    acq_device_list.append("Delphi Controller")
    modaility_list = [Modality.BEHAVIOR_VIDEOS]

if "pirouette" in current_experiment:
    # pirouette free behavior
    pirouette_stim = StimulusEpoch(
        stimulus_name="Pirouette Behavior",
        stimulus_modalities=[StimulusModality.FREE_MOVING],
        stimulus_start_time=expt_start_time,
        stimulus_end_time=expt_end_time,
        code=pirouette_bonsai_acq,
        active_devices=[ephys_assembly.name, "Coaxial Commutator"],
    )

    stim_epochs.append(pirouette_stim)
    acq_code.append(pirouette_bonsai_acq)
    (acq_code.append(lifealert),)
    acq_device_list.append(ephys_assembly.name)
    acq_device_list.append("Onix Breakout Board")
    acq_device_list.append("Coaxial Commutator")
    acq_device_list.append("Harp White Rabbit")
    acq_device_list.append("Harp Output Expander")
    acq_device_list.append("Magnetic Encoder")
    device_configs.append(ephys_assembly_config)

    modaility_list = [Modality.ECEPHYS, Modality.BEHAVIOR_VIDEOS]

# Add cameras to acquistion list
for cam_assembly in cam_assemblies:
    acq_device_list.append(cam_assembly.camera.name)

# create data streams
data_streams = DataStream(
    stream_start_time=expt_start_time,
    stream_end_time=expt_end_time,
    modalities=modaility_list,
    code=acq_code,
    notes="Poking behavior to receieve odors is also considered a BEHAVIOR modality.",
    active_devices=acq_device_list,
    configurations=device_configs,
)


In [59]:
# subject details
if "pirouette" in current_experiment:
    platform_surface = f"{beh_box.name}: {beh_box.internal_material}"
else:
    platform_surface = f"{delphi_cage.name}: {delphi_cage.internal_material}"

print(platform_surface)
subject_details = AcquisitionSubjectDetails(
    mouse_platform_name=platform_surface,
)

Pirouette Behavior Box: Bedding and nesting material


In [60]:
# Build acquisition object
acquisition = Acquisition(
    experimenters=experimenters,
    subject_id=subject_id,
    instrument_id=instrument,
    protocol_id=[protocol],
    acquisition_type=acquisition_type,
    acquisition_start_time=expt_start_time,
    acquisition_end_time=expt_end_time,
    coordinate_system=CoordinateSystemLibrary.BREGMA_ARI,
    data_streams=[data_streams],
    stimulus_epochs=stim_epochs,
    subject_details=subject_details,
)

In [61]:
# Generate acquisition metadata files
if __name__ == "__main__":
    serialized = acquisition.model_dump_json()
    deserialized = Acquisition.model_validate_json(serialized)
    deserialized.write_standard_file(output_directory=metadata_output_path)

### Procedures JSON Mapping

In [31]:
# Import libraries
from aind_data_schema.core.procedures import Procedures
from aind_data_schema.components.subject_procedures import Surgery
from aind_data_schema_models.units import MassUnit, TimeUnit
from aind_data_schema.components.surgery_procedures import (
    Anaesthetic,
    Craniotomy,
    Headframe,
    ProbeImplant,
    CraniotomyType,
    ProtectiveMaterial,
)

In [ ]:
"""Get craniotomy information"""
# craniotomy starting weight
match = re.search(r"Craniotomy:\s*([\d\.]+)", parsed_data.get("weight_before"))
craniotomy_weight_before = float(match.group(1)) if match else None

# craniotomy end weight
match = re.search(r"Craniotomy:\s*([\d\.]+)", parsed_data.get("weight_after"))
craniotomy_weight_after = float(match.group(1)) if match else None

# induction type
match = re.search(r"Craniotomy:\s*([^\s/]+)", parsed_data.get("anesthesia_induction"))
craniotomy_induction = match.group(1) if match else None
craniotomy_anesthetic = Anaesthetic(
    anaesthetic_type=craniotomy_induction,
    duration=210.0,  # hardcoded because a cranotiomy end time doesn't exist in the surgery notes
    duration_unit=TimeUnit.M,
)

craniotomy = Craniotomy(
    craniotomy_type=CraniotomyType.OTHER,
    protective_material=ProtectiveMaterial.KWIK_CAST,
    dura_removed=True,
)

# Create craniotomy object
craniotomy_surgery = Surgery(
    protocol_id=protocol,
    start_date=pd.to_datetime(
        date_expt_df.loc[date_expt_df["experiment"] == "Craniotomy", "date"]
    ).iloc[0],
    experimenters=surgeons,
    animal_weight_prior=craniotomy_weight_before,
    animal_weight_post=craniotomy_weight_after,
    weight_unit=MassUnit.G,
    anaesthesia=craniotomy_anesthetic,
    procedures=[craniotomy],
    notes=parsed_data.get("surgery_notes"),
)

"""Get implantation information"""
# implantation starting weight
match = re.search(r"Implantation:\s*([\d\.]+)", parsed_data.get("weight_before"))
implantation_weight_before = float(match.group(1)) if match else None

# implantation end weight
match = re.search(r"Implantation:\s*([\d\.]+)", parsed_data.get("weight_after"))
implantation_weight_after = float(match.group(1)) if match else None

# 1) Keep only valid HH:MM entries
valid = [t for t in kx_df["time"] if isinstance(t, str) and re.match(r"^\d{1,2}:\d{2}$", t)]

# 2) Convert to (hour, minute) ints
hm = [(int(h), int(m)) for h, m in (s.split(":") for s in valid)]

# 3) Identify values strictly greater than 12:00.
#    Rule: hours == 12 with minute > 0 are after noon; any entries AFTER the first 12:xx
#    with hour in 1..11 are PM (due to sequence order).
minutes_in_day = []  # convert to minutes since midnight
seen_noon = False
for s, (h, m) in zip(valid, hm):
    if h == 12 and m > 0:
        # Noon boundary
        minutes_in_day.append(h * 60 + m)
        seen_noon = True
    elif seen_noon and 1 <= h <= 11:
        # After noon
        minutes_in_day.append((h + 12) * 60 + m)
    else:
        # Before noon
        minutes_in_day.append(h * 60 + m)

implantation_induction_duration = np.max(minutes_in_day) - np.min(minutes_in_day)

# implantation anesthetic
implantation_anesthetic = Anaesthetic(
    anaesthetic_type="".join(kx_df["agent"].unique()),
    duration=implantation_induction_duration,  # hardcoded because a cranotiomy end time doesn't exist in the surgery notes
    duration_unit=TimeUnit.M,
)

probe_implant = ProbeImplant(
    implanted_device=probe1,
    device_config=probe_config,
)

# Create implantation object
implantation_surgery = Surgery(
    protocol_id=protocol,
    start_date=pd.to_datetime(
        date_expt_df.loc[date_expt_df["experiment"] == "Implantation", "date"]
    ).iloc[0],
    experimenters=surgeons,
    animal_weight_prior=implantation_weight_before,
    animal_weight_post=implantation_weight_after,
    weight_unit=MassUnit.G,
    anaesthesia=implantation_anesthetic,
    procedures=[probe_implant],
    notes=parsed_data.get("implantation_notes"),
)

# Create headframe object
headframe = Headframe(
    headframe_type="3D printed custom headframe for chronic implantation",
    headframe_part_number="CAD model: Headplate_mark12",
)

# Ground screw  -- for just myomatirx
# gnd_screw = GroundWireImplant(
#     ground_electrode_location=MouseAnatomy.CEREBELLUM,
#     ground_wire_material=GroundWireMaterial.SILVER,
#     ground_wire_diameter=0.0060,
#     ground_wire_diameter_unit=SizeUnit.IN,
# )


# Headframe surgery date
match = re.search(r"(\d{1,2}/\d{1,2}/\d{2,4})", parsed_data.get("surgery_history"))
date_str = match.group(1)
# Parse into datetime object (assuming MM/DD/YY format)
# headframe_surgery_date = datetime.strptime(date_str, "%m/%d/%y")
headframe_surgery_date = datetime.strptime("07/01/25", "%m/%d/%y")

headframe_surgery = Surgery(
    protocol_id=protocol,
    start_date=headframe_surgery_date,
    experimenters=[surgeons[1]],
    procedures=[headframe],
)


In [37]:
# Create procedures object from surgery notes  -- Delphi
procedures = Procedures(
    subject_id=subject,
)

In [38]:
# Create procedures object from surgery notes
procedures = Procedures(
    subject_id=subject,
    subject_procedures=[craniotomy_surgery, implantation_surgery, headframe_surgery],
    coordinate_system=CoordinateSystemLibrary.BREGMA_ARI,
    notes=parsed_data.get("overall_impressions"),
)

In [39]:
# Create JSON
# Generate acquisition metadata files
if __name__ == "__main__":
    serialized = procedures.model_dump_json()
    deserialized = Procedures.model_validate_json(serialized)
    deserialized.write_standard_file(output_directory=metadata_output_path)

### Subject JSON

Automatic generation of subject.json:

http://aind-metadata-service/api/v2/subject/801055 <-- replace with your mouse ID
